# Model size: Qwen3.5-9B against Qwen3.5-2B on the computing tasks with the gold sheet (exploratory)

`experiments/2026-11-25-concise-math-cot/README.md`, section "Model size". The same 30 development questions
(`model_size_ids.txt`: 10 per computing task, explore200's, seed 0) and the same prompts, frames (16, `uniform128`),
greedy decoding, 2,048 new tokens and answer rule as the 2B runs; thinking off. The 9B is loaded in bf16 with
`device_map="auto"`, no quantization, and answers one question at a time (batch size 1; the 2B keeps its batches of up
to 8); if any weight would leave the GPU the run stops.

| Run, in this order | Model | Condition | `followup` name |
|---|---|---|---|
| 1 | 9B | C: gold sheet, direct | `gold_csv` |
| 2 | 9B | F1: gold sheet, concise CoT | `cot_f1` |
| 3 | 9B | E: gold sheet + relations, direct | `gold_relations` |
| 4 | 2B | F1 (the smoke stage's questions are skipped: already answered) | `cot_f1` |
| 5 | 9B | A: video only, direct | `A_video` |
| 6 (only with `RUN_F2 = True`) | 9B | F2: + formulas | `cot_f2` |

The 2B's A, C, D and E are the structured-CoT pilot's predictions.

**Colab:** Runtime > Change runtime type > **A100 GPU** (or **L4**: 24 GB, slower, and the 9B's fit there is what
the smoke stage checks). Upload `mvp-3d-vlm-cot-9b.zip` to `MyDrive/mvp/`.

- **`STAGE = "smoke"`** (default): 1 question per task, every 9B condition (12 replies). It first checks the 2B F3 smoke
  replies for copying from the worked example, then loads the 9B once on its own (memory, load time), then runs.
  It prints the peak GPU memory, seconds per question for each condition and an estimate for the main stage. Decide
  there whether to go on.
- **`STAGE = "main"`**: the 30 questions, runs 1–5 (6 with `RUN_F2 = True`). Resumable: run it again from the top
  after a disconnect; answered questions are skipped.

Answers are written to Drive as they come (`MyDrive/mvp/cot_followup/<RUN_NAME>/`, the 9B's in `qwen3.5-9b/`): the
9B's after each question, the 2B's after each batch. At the end the folder is
zipped to `MyDrive/mvp/cot_9b_results_<STAGE>.zip`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-cot-9b.zip"
RUN_NAME = "dev_v1"                      # the 2B smoke run's folder: its F1 answers are reused, its F3 replies checked
RESULTS_DIR = f"{DRIVE_DIR}/cot_followup/{RUN_NAME}"
STAGE = "smoke"                          # smoke (1 question per task) | main (10 per task, 30)
RUN_F2 = False                           # 9B F2, last, only if time allows
SHOW_PROMPTS = STAGE == "smoke"          # print the text after the video with each reply
SMALL, LARGE = "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-9B"
LARGE_BATCH = 1                          # the 9B: one question at a time
SEED = 0
PROMPT_VERSION = "2026-11-25.1"          # must be cot_variants.PROMPT_VERSION in the code zip
VSI_REPO, VSI_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"
INDEX_SHA256 = "22e0ee1ed3400a5ca59bc49c3791f433265115b4301c183b9772e4e014f5573c"  # build_gold_sheets.py, all 288 scenes
OBJECTS_SHA256 = "bc0214c926be913606031474df123b2d6ee749fba39f799d56f4e8b1b21f41a5"

import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from google.colab import drive

assert STAGE in ("smoke", "main"), STAGE
drive.mount("/content/drive")
CODE = Path("/content/mvp-3d-vlm")
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(CODE)
    CODE_COMMIT = z.comment.decode() or "unknown"
X = CODE / "experiments/2026-11-25-concise-math-cot"
assert (X / "model_size_ids.txt").is_file(), f"{CODE_ZIP} has no model_size_ids.txt: not the 9B code"
reqs = [l.split("#")[0].strip() for l in (CODE / "requirements.txt").read_text().splitlines()]
reqs = [r for r in reqs if r and not r.startswith("--extra-index-url") and r.split("==")[0] not in ("torch", "torchvision")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(CODE)], check=True)
sys.path[:0] = [str(CODE / "src"), str(X)]
ENV = {**os.environ, "PYTHONPATH": str(CODE / "src")}
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 GPU or L4"
GPU = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert "A100" in GPU or "L4" in GPU, f"{GPU}: this notebook is planned for an A100 or an L4"
from mvp.benchmarks import cot_variants
assert cot_variants.PROMPT_VERSION == PROMPT_VERSION, f"the code's prompts are version {cot_variants.PROMPT_VERSION}"
import followup, analysis
K = followup.STAGES["smoke" if STAGE == "smoke" else "extended"]
LARGE_DIR = Path(followup.model_dir(RESULTS_DIR, LARGE))
assert LARGE_DIR.name == analysis.LARGE_DIR
ORDER = [(LARGE, "gold_csv"), (LARGE, "cot_f1"), (LARGE, "gold_relations"), (SMALL, "cot_f1"), (LARGE, "A_video")]
ORDER += [(LARGE, "cot_f2")] if RUN_F2 else []
if STAGE == "smoke":
    ORDER = [(m, c) for m, c in ORDER if m == LARGE]  # the 2B's F1 smoke answers exist already
print(f"code {CODE_COMMIT}; GPU {GPU}, {GPU_GB:.0f} GB; {STAGE}: {K} per task, {3 * K} questions; runs {ORDER}")

## Questions, gold sheets and videos

`test.jsonl` at the pinned revision; the gold sheets rebuilt and their hashes checked; the stage's questions, which
must be `model_size_ids.txt`'s (the smoke stage: the first of each task); their videos; and a check that the eval shows
each question the frames the pilot's runs saw.

In [ ]:
import numpy as np
from huggingface_hub import HfFileSystem, hf_hub_download

RAW, VGGT = CODE / "data/raw/VSI-Bench", CODE / "data/vggt"
RAW.mkdir(parents=True, exist_ok=True)
shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset", revision=VSI_REVISION), RAW / "test.jsonl")
assert hashlib.sha256((RAW / "test.jsonl").read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"
SHEETS = CODE / "tasks/openvocab_adapter/out_full"
r = subprocess.run([sys.executable, str(CODE / "experiments/2026-11-19-gold-sheet-relations/build_gold_sheets.py"),
                    "--questions", str(RAW / "test.jsonl"), "--prompts",
                    str(CODE / "experiments/2026-11-19-gold-sheet-relations/prompts_full.jsonl"), "--out", str(SHEETS)],
                   cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0 and f"index.csv sha256 {INDEX_SHA256}" in r.stdout and f"objects.csv sha256 {OBJECTS_SHA256}" in r.stdout, \
    "the gold sheets are not the expected ones: " + r.stderr[-2000:]

QUESTIONS = followup.load_questions(RAW / "test.jsonl")
pool = followup.dev_pool(QUESTIONS)
IDS = followup.nested_subset(pool, QUESTIONS, K, SEED)
LISTED = [int(x) for x in (X / "model_size_ids.txt").read_text().split()]
assert set(IDS) <= set(LISTED) and (STAGE == "smoke" or sorted(IDS) == sorted(LISTED)), "not model_size_ids.txt's questions"
SCENES = sorted({f"{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}" for q in IDS})
fs = HfFileSystem()
for dataset in sorted({s.split("/")[0] for s in SCENES}):
    todo = [s for s in SCENES if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]
    if todo:
        with fs.open(f"datasets/{VSI_REPO}@{VSI_REVISION}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in todo:
                (RAW / s).parent.mkdir(parents=True, exist_ok=True)
                (RAW / f"{s}.mp4").write_bytes(z.read(f"{s}.mp4"))
from decord import VideoReader, cpu
for s in SCENES:  # frame_indices.npy as the VGGT export takes them: min(128, N) frames by np.linspace
    n = len(VideoReader(str(RAW / f"{s}.mp4"), ctx=cpu(0)))
    (VGGT / s).mkdir(parents=True, exist_ok=True)
    np.save(VGGT / s / "frame_indices.npy", np.linspace(0, n - 1, num=min(128, n), dtype=int).astype(np.int32))
from mvp.eval.vsibench import frame_indices
pilot = {r["id"]: r for r in map(json.loads, (CODE / "results/zeroshot/vsibench/structured_cot/explore200/gold_cot.jsonl").read_text().splitlines())}
video = lambda q: f"{RAW}/{QUESTIONS[q]['dataset']}/{QUESTIONS[q]['scene_name']}.mp4"
bad = [q for q in IDS if frame_indices(video(q), "uniform128", 16, VGGT) != pilot[q]["frame_indices"]]
assert not bad, f"the eval would show other frames than the pilot's runs for {bad}"
print(f"{len(IDS)} questions {IDS}, {len(SCENES)} videos; the frames are the pilot's")

## The 2B's F3 smoke replies: copied from the worked example?

CPU only. For each 2B F1–F3 reply in `RUN_NAME`: the answer read, its score, whether it is the example's answer
(1.059 m, 183 cm) and which of the example's numbers it quotes that its own sheet does not have. F1 and F2 never see
the example: they are the control. Then F3's replies in full.

In [ ]:
small = {c: Path(RESULTS_DIR) / stem / "predictions.jsonl" for c, (stem, _) in analysis.VARIANTS.items()}
small = {c: analysis.load_run(p) for c, p in small.items() if p.is_file()}
if "F3" not in small:
    print(f"no 2B F3 replies in {RESULTS_DIR}: nothing to check")
else:
    errors = analysis.error_rows(small, SHEETS, conditions=tuple(small))
    cols = ["condition", "id", "question_type", "ground_truth", "read", "score", "copies_example_answer",
            "quotes_example_numbers", "example_numbers_quoted", "cut_off", "unread"]
    print(errors[cols].to_string(index=False))
    for q, row in small["F3"].iterrows():
        print("=" * 100, f"\nF3 | question {q} ({row['question_type']}) | truth {row['ground_truth']} | score {row['score']:.2f}")
        print(row["prediction"])

## The 9B: download, and one load on its own (smoke stage)

The weights are downloaded once (about 19 GB). Then the 9B is loaded in bf16 with `device_map="auto"` in a separate
process: the load time, the memory its weights take and where accelerate put them. If any part would go to the CPU or
disk, `QwenVL` stops: the 9B does not fit, and nothing is run in another precision.

In [ ]:
from huggingface_hub import snapshot_download
t0 = time.time()
snapshot_download(LARGE)
print(f"{LARGE} downloaded in {(time.time() - t0) / 60:.1f} min")
LOAD_LOG = LARGE_DIR / "load_check.json"
if STAGE == "smoke" or not LOAD_LOG.is_file():
    probe = f"""
import json, time, torch
t = time.time()
from mvp.models.qwen_vl import QwenVL
m = QwenVL({LARGE!r}, device_map="auto")
torch.cuda.synchronize()
print(json.dumps({{"load_seconds": round(time.time() - t, 1), "weights_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
                  "devices": sorted({{str(d) for d in m.model.hf_device_map.values()}}), "dtype": str(m.model.dtype)}}))
"""
    r = subprocess.run([sys.executable, "-c", probe], env=ENV, capture_output=True, text=True)
    assert r.returncode == 0, "the 9B did not load in bf16 on this GPU; stop here and report:\n" + r.stderr[-3000:]
    LOAD = {**json.loads(r.stdout.strip().splitlines()[-1]), "gpu": GPU, "gpu_gb": round(GPU_GB, 1)}
    LARGE_DIR.mkdir(parents=True, exist_ok=True)
    LOAD_LOG.write_text(json.dumps(LOAD, indent=1))
LOAD = json.loads(LOAD_LOG.read_text())
print(LOAD)

## Runs

In the order above. `followup.run` checks each folder's configuration (another model, prompt, setting, sheet or seed
stops it), logs the exact prompts, answers only the questions not answered yet and times the call (the model's
loading included). A thread reads `nvidia-smi` every second for the peak GPU memory. Each call is logged to
`<RUN_NAME>/qwen3.5-9b/stage_log.jsonl`. An error (out of memory included) stops the notebook: report it.

In [ ]:
class PeakGPU:
    """The highest memory.used nvidia-smi reports while the block runs, in GB (the eval runs in another process)."""
    def __enter__(self):
        self.peak, self.done = 0, threading.Event()
        self.thread = threading.Thread(target=self.poll, daemon=True)
        self.thread.start()
        return self
    def poll(self):
        while not self.done.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True).stdout
            self.peak = max(self.peak, int(out.split()[0]) / 1024)
            self.done.wait(1)
    def __exit__(self, *exc):
        self.done.set()
        self.thread.join()

LOG = LARGE_DIR / "stage_log.jsonl"
for model, cond in ORDER:
    t0 = time.time()
    with PeakGPU() as gpu:
        run_dir, cmd, todo = followup.run(cond, K, SHEETS, RESULTS_DIR, pool, SEED, RAW / "test.jsonl", model,
                                          vggt_root=VGGT, prompt_version=PROMPT_VERSION, env=ENV,
                                          device_map="auto" if model == LARGE else followup.DEVICE_MAP,
                                          batch_size=LARGE_BATCH if model == LARGE else None)
    secs = time.time() - t0
    rec = {"stage": STAGE, "model": model, "condition": cond, "new": len(todo), "seconds": round(secs, 1),
           "peak_gpu_gb": round(gpu.peak, 1), "gpu": GPU, "code": CODE_COMMIT}
    with open(LOG, "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{model.split('/')[-1]} {cond}: {len(todo)} new in {secs / 60:.1f} min, peak GPU {gpu.peak:.1f} GB -> {run_dir}")

## Time and memory (and, after the smoke stage, the estimate for the main stage)

Per call: peak GPU memory and seconds per question, with and without the model's loading (the load check's time).
The estimate for the main stage: per 9B condition, one load plus 27 new questions at the smoke stage's rate; the 2B's
F1 from its own smoke timing; F2 at F1's rate if `RUN_F2`. A rough guide: three questions per condition are few. The
9B answers one question at a time in both stages, so its peak memory here holds for the main stage.

In [ ]:
log = [json.loads(l) for l in LOG.read_text().splitlines() if json.loads(l)["stage"] == STAGE and json.loads(l)["new"]]
load = LOAD["load_seconds"]
rate = {}
for r in log:
    per = r["seconds"] / r["new"]
    rate[(r["model"], r["condition"])] = max(0.0, r["seconds"] - load) / r["new"] if r["model"] == LARGE else per
    print(f"{r['model'].split('/')[-1]:>10} {r['condition']:>15}: {r['new']} questions, peak GPU {r['peak_gpu_gb']:.1f} GB, "
          f"{per:.0f} s per question with the load, {rate[(r['model'], r['condition'])]:.0f} without")
if STAGE == "smoke":
    n_new = 30 - 3
    est = {c: load + n_new * rate[(LARGE, c)] for c in ("gold_csv", "cot_f1", "gold_relations", "A_video") if (LARGE, c) in rate}
    est["cot_f2 (if RUN_F2)"] = load + n_new * rate.get((LARGE, "cot_f2"), rate.get((LARGE, "cot_f1"), 0))
    t2 = Path(RESULTS_DIR) / "cot_f1" / "timing.jsonl"
    small_rate = [json.loads(l)["seconds_per_new_question"] for l in t2.read_text().splitlines()] if t2.is_file() else []
    if small_rate:  # the 2B smoke's seconds per question include its load: an upper bound
        est["2B cot_f1"] = n_new * small_rate[0]
    for c, s in est.items():
        print(f"main stage, {c}: about {s / 60:.0f} min")
    core = sum(s for c, s in est.items() if not c.startswith("cot_f2"))
    print(f"main stage, runs 1-5: about {core / 60:.0f} min (+ setup about 10 min; + F2 about "
          f"{est['cot_f2 (if RUN_F2)'] / 60:.0f} min)" + ("" if small_rate else "; no 2B F1 timing found, not included"))

## Prompts and replies, for reading

In [ ]:
from mvp.benchmarks.vsibench import final_letter, final_number, MCA_QUESTION_TYPES
for model, cond in ORDER:
    d = Path(followup.model_dir(RESULTS_DIR, model)) / cond
    prompts = {r["id"]: r for r in map(json.loads, (d / "prompts.jsonl").read_text().splitlines())}
    preds = {r["id"]: r for r in map(json.loads, (d / "predictions.jsonl").read_text().splitlines())}
    for q in IDS:
        p = preds.get(q)
        print("=" * 100)
        print(f"{model.split('/')[-1]} {cond} | question {q} ({QUESTIONS[q]['question_type']}) | truth {QUESTIONS[q]['ground_truth']}")
        if SHOW_PROMPTS:
            print("--- prompt after the video ---\n" + prompts[q]["after_video"])
        if p is None:
            print("--- (no reply yet) ---")
            continue
        read = final_letter(p["prediction"]) if p["question_type"] in MCA_QUESTION_TYPES else final_number(p["prediction"])
        print(f"--- reply ({p.get('n_new_tokens')} tokens{', CUT OFF' if p.get('hit_token_limit') else ''}) ---\n"
              f"{p['prediction']}\n--- read: {read}; score {p['score']:.2f}")

## Analysis (exploratory) and the results zip

`analysis.py --model-size`: each model's conditions on the questions all of them answered, and the contrasts (9B F1 −
9B C, 2B F1 − 2B C, their difference, 9B C − 2B C, 9B E − 2B E) with sign tests, Holm and paired bootstrap intervals.
Then the whole `RUN_NAME` folder (the 2B's runs too) is zipped to Drive.

In [ ]:
text, runs, rows = analysis.size_report(RESULTS_DIR)
print(text)
archive = shutil.make_archive(f"{DRIVE_DIR}/cot_9b_results_{STAGE}", "zip", RESULTS_DIR)
print(f"saved {RESULTS_DIR}/model_size.md; zipped {RESULTS_DIR} to {archive} "
      f"({os.path.getsize(archive) / 1e6:.1f} MB); code {CODE_COMMIT}")